# **Logistics Operations RAG Assistant**

**Group: swateek**

## **Objective**

The main objective of this assignment is to process and analyse a collection of logistics and operational documents from AeroVelo Logistics Global to prepare them for implementing a **Retrieval-Augmented Generation (RAG)** system. This involves:

* Understand the Logistics Data: Gain an understanding of the structure, content, and context of the provided documents.
* Perform Exploratory Analysis: Analyse the documents to understand their length, distribution, and content.
* Prepare Documents for Retrieval: Clean and split the documents into smaller chunks suitable for semantic retrieval.
* Build a RAG Knowledge Base: Generate embeddings and store document chunks in a vector database.
* Retrieve Relevant Information: Retrieve the most relevant document chunks for a user query.
* Generate Grounded Responses: Use a Large Language Model (LLM) to answer questions based on the retrieved context.
* Evaluate the RAG System: Assess retrieval quality, answer correctness, grounding, and unsupported question handling.

### **Business Value**

The project aims to use RAG to improve access to logistics and operational information within AeroVelo Logistics Global. The key business objectives include:

* Faster Information Retrieval: <br>
Reduce the time required to search across logistics, supply chain, finance, governance, compliance, legal, and HR documents.

* Improved Operational Support: <br>
Provide relevant information for supplier, procurement, incident management, and operational queries.

* Consistent Responses: <br>
Generate answers grounded in approved organisational documents rather than relying only on the LLM's internal knowledge.

* Reduced Manual Search Effort: <br>
Allow users to query multiple documents through a single assistant instead of manually reviewing individual files.

* Better Handling of Unsupported Queries: <br>
Identify when the available document collection does not contain sufficient information to answer a question.

## **1. Data Loading, Preparation and Analysis** <font color=red>[40 marks]</font>

### **1.1 Data Understanding**

The dataset contains documents from AeroVelo Logistics Global covering different operational and business functions.

The documents are organised into the following folders:

- `compliance`: regulatory and compliance-related documents
- `finance`: financial authorisation and procurement-related documents
- `governance`: regional governance information
- `hr`: human resources policies
- `legal`: legal and supplier data-processing documents
- `operations`: logistics incident management procedures
- `pdfs`: documents available in PDF format
- `supply_chain`: supplier qualification and Bill of Materials information

The dataset contains both Markdown (`.md`) and PDF (`.pdf`) files. These documents will be loaded and converted into LangChain document objects while preserving their source information as metadata.

### **1.2 Load and Preprocess the Data** <font color=red>[12 marks]</font>

#### Loading libraries

In [1]:
# Import essential libraries

import os
import re
import shutil
from collections import Counter
from pathlib import Path

import nltk
import pandas as pd
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

NOTEBOOK_NAME = "Logistics_Operations_RAG_Assistant_Starter.ipynb"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CHAT_MODEL_NAME = "gemini-3.5-flash-lite"
COLLECTION_NAME = "logistics_rag"
CHUNK_SIZE = 800
CHUNK_OVERLAP = 120
RETRIEVAL_K = 4
MAX_OUTPUT_TOKENS = 512


def notebook_dir() -> Path:
    """Directory that contains this notebook, independent of the kernel working directory."""
    cwd = Path.cwd().resolve()
    if (cwd / NOTEBOOK_NAME).is_file():
        return cwd
    for parent in [cwd, *cwd.parents]:
        if (parent / NOTEBOOK_NAME).is_file():
            return parent
        nested = (
            parent / "practice-capstone-project" / "swateek" / "results" / NOTEBOOK_NAME
        )
        if nested.is_file():
            return nested.parent
    return cwd


NOTEBOOK_DIR = notebook_dir()
DATASET_DIR = (NOTEBOOK_DIR / ".." / "dataset").resolve()
CHROMA_DIR = (NOTEBOOK_DIR / ".." / "chroma" / "logistics_rag").resolve()


def load_api_key() -> None:
    """Load the first .env found above the working directory or the notebook directory."""
    search_starts = [Path.cwd().resolve(), NOTEBOOK_DIR]
    seen = set()
    loaded_from = None
    for start in search_starts:
        for folder in [start, *start.parents]:
            if folder in seen:
                continue
            seen.add(folder)
            env_path = folder / ".env"
            if env_path.is_file():
                load_dotenv(env_path, override=False)
                loaded_from = env_path
                break
        if loaded_from is not None:
            break
    if not os.environ.get("GOOGLE_API_KEY") and os.environ.get("GEMINI_API_KEY"):
        os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]
    if loaded_from is not None:
        print(f"Loaded environment from {loaded_from}")
    else:
        print("No .env file found while walking parent directories.")
    if os.environ.get("GOOGLE_API_KEY"):
        print("GOOGLE_API_KEY is set.")
    else:
        print(
            "GOOGLE_API_KEY is not set. Generation cells will fail until a .env provides it."
        )


load_api_key()
print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Dataset directory: {DATASET_DIR}")

/var/folders/1d/fx2jbk3s51143jzgrm32vzc00000gn/T/ipykernel_24444/1705515949.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader


Loaded environment from /Users/swateek/Documents/codebase/misc/swateek-github/upgrad-capstone/practice-capstone-project/swateek/results/.env
GOOGLE_API_KEY is set.
Notebook directory: /Users/swateek/Documents/codebase/misc/swateek-github/upgrad-capstone/practice-capstone-project/swateek/results
Dataset directory: /Users/swateek/Documents/codebase/misc/swateek-github/upgrad-capstone/practice-capstone-project/swateek/dataset


#### **1.2.1 Load Logistics Documents** <font color=red>[7 marks]</font>

Load all Markdown (`.md`) and PDF (`.pdf`) documents from the dataset folder. Store useful metadata such as the file name, business category, file type, and source path for later retrieval and evaluation.

In [2]:
# Load all Markdown and PDF files from the dataset folder


def discover_document_paths(dataset_dir: Path) -> list[Path]:
    paths = [
        path
        for path in dataset_dir.rglob("*")
        if path.is_file() and path.suffix.lower() in {".md", ".pdf"}
    ]
    return sorted(paths, key=lambda path: path.relative_to(dataset_dir).as_posix())


document_paths = discover_document_paths(DATASET_DIR)
print(f"Discovered {len(document_paths)} files under {DATASET_DIR}")
for path in document_paths:
    print(path.relative_to(DATASET_DIR).as_posix())

Discovered 10 files under /Users/swateek/Documents/codebase/misc/swateek-github/upgrad-capstone/practice-capstone-project/swateek/dataset
compliance/SOP-COM-015_Regulatory_Breach_Report.md
finance/FIN-AUTH-101_Authorization_Framework.md
governance/RGM-2026_Regional_Governance_Matrix.md
hr/HR-POL-030-EMEA_Remote_Work.md
hr/HR-POL-030_Remote_Work_US.md
legal/LEG-GDPR-007_Supplier_Data_Processing.md
operations/OP-204_Incident_Management_v4.md
pdfs/RGM-2026_Regional_Governance_Matrix.pdf
supply_chain/AVL-PUR-012_Supplier_Qualification.md
supply_chain/BOM-FLT-H2NEX.md


In [3]:
# Load the logistics documents and store useful metadata


def load_logistics_documents(paths: list[Path], dataset_dir: Path) -> list[Document]:
    loaded = []
    for path in paths:
        relative = path.relative_to(dataset_dir)
        file_type = path.suffix.lower().lstrip(".")
        metadata = {
            "file_name": path.name,
            "category": path.parent.name,
            "file_type": file_type,
            "source_path": relative.as_posix(),
        }
        if file_type == "md":
            page_content = (
                TextLoader(str(path), encoding="utf-8").load()[0].page_content
            )
        elif file_type == "pdf":
            pages = PyPDFLoader(str(path)).load()
            page_content = "\n\n".join(page.page_content for page in pages)
        else:
            continue
        loaded.append(Document(page_content=page_content, metadata=metadata))
    return loaded


documents = load_logistics_documents(document_paths, DATASET_DIR)
print(f"Loaded {len(documents)} documents")
if len(documents) != 10:
    raise RuntimeError(f"Expected 10 documents, found {len(documents)}")
required_metadata = {"file_name", "category", "file_type", "source_path"}
for document in documents:
    missing = required_metadata - set(document.metadata)
    if missing:
        raise RuntimeError(
            f"Missing metadata {missing} on {document.metadata.get('source_path')}"
        )

Loaded 10 documents


In [4]:
# Inspect the first loaded document

first_document = documents[0]
print("Metadata:")
for key in ("file_name", "category", "file_type", "source_path"):
    print(f"  {key}: {first_document.metadata[key]}")
print("\nContent preview:")
print(first_document.page_content[:500])

Metadata:
  file_name: SOP-COM-015_Regulatory_Breach_Report.md
  category: compliance
  file_type: md
  source_path: compliance/SOP-COM-015_Regulatory_Breach_Report.md

Content preview:
# AeroVelo Logistics Global — Regulatory Breach Reporting

**Document ID:** SOP-COM-015  
**Version:** 2.0 (Current)  
**Effective Date:** 2025-03-01  
**Classification:** Internal — Compliance Officers and Above  
**Owner:** Global Compliance Directorate  
**Approved By:** General Counsel and Board Risk Committee

---

## 1. Purpose

This procedure defines the reporting obligations, escalation timelines, and remediation requirements for regulatory breaches across all AVL operating jurisdictions


#### **1.2.2 Clean and Normalise Document Text** <font color=red>[5 marks]</font>

Clean the loaded document text by removing unnecessary whitespace and formatting inconsistencies while preserving the original document meaning and metadata.

In [5]:
# Clean and normalise the document text


def normalise_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


cleaned_documents = [
    Document(
        page_content=normalise_text(document.page_content),
        metadata=dict(document.metadata),
    )
    for document in documents
]
print(f"Cleaned corpus length: {len(cleaned_documents)}")
corpus_text = "\n".join(document.page_content for document in cleaned_documents)
for token in ("$10,000", "2,800 psi"):
    print(f"Corpus still contains {token!r}: {token in corpus_text}")

Cleaned corpus length: 10
Corpus still contains '$10,000': True
Corpus still contains '2,800 psi': True


### **1.3 Exploratory Data Analysis** <font color=red>[16 marks]</font>

#### **1.3.1 Analyse Document Lengths** <font color=red>[4 marks]</font>

Calculate the average, maximum and minimum document length.

In [6]:
# Calculate the average, maximum and minimum document length

length_rows = []
for document in cleaned_documents:
    length_rows.append(
        {
            "file_name": document.metadata["file_name"],
            "characters": len(document.page_content),
            "words": len(document.page_content.split()),
        }
    )

length_table = pd.DataFrame(length_rows).sort_values("file_name").reset_index(drop=True)
character_lengths = length_table["characters"]
print(length_table.to_string(index=False))
print()
print(f"Minimum characters: {int(character_lengths.min())}")
print(f"Maximum characters: {int(character_lengths.max())}")
print(f"Average characters: {character_lengths.mean():.1f}")

                               file_name  characters  words
   AVL-PUR-012_Supplier_Qualification.md        2836    395
                        BOM-FLT-H2NEX.md        5063    720
 FIN-AUTH-101_Authorization_Framework.md        5080    689
          HR-POL-030-EMEA_Remote_Work.md        5605    785
            HR-POL-030_Remote_Work_US.md        3955    542
LEG-GDPR-007_Supplier_Data_Processing.md        4737    647
        OP-204_Incident_Management_v4.md        4146    559
  RGM-2026_Regional_Governance_Matrix.md        4973    660
 RGM-2026_Regional_Governance_Matrix.pdf        1395    186
 SOP-COM-015_Regulatory_Breach_Report.md        2985    396

Minimum characters: 1395
Maximum characters: 5605
Average characters: 4077.5


#### **1.3.2 Analyse Word Frequencies** <font color=red>[6 marks]</font>

Analyse the frequency of words in the logistics documents and identify the most and least frequently occurring words.

In [7]:
# Analyse word frequencies in the logistics documents

try:
    stop_words = set(nltk.corpus.stopwords.words("english"))
except LookupError:
    nltk.download("stopwords", quiet=True)
    stop_words = set(nltk.corpus.stopwords.words("english"))

token_pattern = re.compile(r"[a-z]{2,}")
token_counts = Counter()
for document in cleaned_documents:
    tokens = token_pattern.findall(document.page_content.lower())
    token_counts.update(token for token in tokens if token not in stop_words)

most_frequent = token_counts.most_common(20)
least_frequent = sorted(token_counts.items(), key=lambda item: (item[1], item[0]))[:20]

print("20 most frequent non-stop words:")
for word, count in most_frequent:
    print(f"  {word}: {count}")

print("\n20 least frequent non-stop words (ties broken alphabetically):")
for word, count in least_frequent:
    print(f"  {word}: {count}")

/Users/swateek/Documents/codebase/misc/swateek-github/upgrad-capstone/practice-capstone-project/swateek/results/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


20 most frequent non-stop words:
  regional: 67
  avl: 55
  remote: 54
  document: 46
  operations: 40
  global: 36
  emea: 34
  incident: 34
  logistics: 29
  work: 29
  aerovelo: 28
  nex: 28
  must: 27
  tier: 27
  compliance: 25
  governance: 25
  hours: 24
  per: 24
  annual: 24
  management: 23

20 least frequent non-stop words (ties broken alphabetically):
  acceptable: 1
  acceptance: 1
  accesses: 1
  accessories: 1
  adaptation: 1
  additionally: 1
  adherence: 1
  administrative: 1
  affected: 1
  africa: 1
  aggregate: 1
  agvs: 1
  alc: 1
  allocated: 1
  allowed: 1
  alternative: 1
  amount: 1
  analyzes: 1
  ancillary: 1
  angeles: 1


#### **1.3.3 Compare Document Similarity** <font color=red>[6 marks]</font>

Analyse the similarity between different logistics documents using TF-IDF vectors and cosine similarity.

In [8]:
# Transform documents using TF-IDF and compute document similarity

file_names = [document.metadata["file_name"] for document in cleaned_documents]
tfidf_matrix = TfidfVectorizer().fit_transform(
    document.page_content for document in cleaned_documents
)
similarity_matrix = cosine_similarity(tfidf_matrix)
similarity_frame = pd.DataFrame(similarity_matrix, index=file_names, columns=file_names)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda value: f"{value:.3f}")
print(similarity_frame.to_string())

rgm_names = [name for name in file_names if "RGM-2026" in name]
if len(rgm_names) == 2:
    score = similarity_frame.loc[rgm_names[0], rgm_names[1]]
    print(
        "\nExpected high similarity between the RGM Markdown and PDF copies "
        f"({rgm_names[0]} vs {rgm_names[1]}): {score:.3f}"
    )

                                          SOP-COM-015_Regulatory_Breach_Report.md  FIN-AUTH-101_Authorization_Framework.md  RGM-2026_Regional_Governance_Matrix.md  HR-POL-030-EMEA_Remote_Work.md  HR-POL-030_Remote_Work_US.md  LEG-GDPR-007_Supplier_Data_Processing.md  OP-204_Incident_Management_v4.md  RGM-2026_Regional_Governance_Matrix.pdf  AVL-PUR-012_Supplier_Qualification.md  BOM-FLT-H2NEX.md
SOP-COM-015_Regulatory_Breach_Report.md                                     1.000                                    0.262                                   0.310                           0.187                         0.148                                     0.336                             0.327                                    0.335                                  0.119             0.109
FIN-AUTH-101_Authorization_Framework.md                                     0.262                                    1.000                                   0.498                           0.168        

### **1.4 Document Creation and Chunking** <font color=red>[12 marks]</font>

#### **1.4.1 Create Overlapping Document Chunks** <font color=red>[12 marks]</font>

Perform appropriate steps to split the logistics documents into smaller overlapping chunks while preserving the document metadata.

In [9]:
# Define a function to split documents into overlapping chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
)


def split_documents(docs: list[Document]) -> list[Document]:
    chunks = text_splitter.split_documents(docs)
    for chunk in chunks:
        chunk.metadata = {
            "file_name": chunk.metadata.get("file_name"),
            "category": chunk.metadata.get("category"),
            "file_type": chunk.metadata.get("file_type"),
            "source_path": chunk.metadata.get("source_path"),
        }
    return chunks

In [10]:
# Process all documents and generate chunks

chunks = split_documents(cleaned_documents)

In [11]:
# Check the total number of chunks

print(f"Total chunks: {len(chunks)}")
print(
    f"Chunk count greater than document count: {len(chunks) > len(cleaned_documents)}"
)

Total chunks: 68
Chunk count greater than document count: True


In [ ]:
# Inspect a sample chunk

sample_chunk = chunks[0]
print(sample_chunk.page_content)
print("\nMetadata:")
for key in ("file_name", "category", "file_type", "source_path"):
    print(f"  {key}: {sample_chunk.metadata[key]}")

## **2. Vector Database and RAG Chain Creation** <font color=red>[35 marks]</font>

### **2.1 Vector Embeddings and Database Creation** <font color=red>[16 marks]</font>

#### **2.1.1 Initialise the Embedding Model** <font color=red>[5 marks]</font>

Initialise an embedding function to generate vector representations of the document chunks.

Initialise an embedding model to convert the document chunks into numerical vector representations for storage and similarity search in the vector database.

In [12]:
# Initialise the embedding model

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15245.56it/s]


Embedding model: sentence-transformers/all-MiniLM-L6-v2


#### **2.1.2 Create and Populate the Vector Database** <font color=red>[11 marks]</font>

Load the document chunk embeddings into a Chroma vector database for semantic retrieval.

In [13]:
# Create a function to add chunks to the vector database


def build_vector_store(chunk_documents: list[Document]) -> Chroma:
    if CHROMA_DIR.exists():
        shutil.rmtree(CHROMA_DIR)
    CHROMA_DIR.mkdir(parents=True, exist_ok=True)
    return Chroma.from_documents(
        documents=chunk_documents,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=str(CHROMA_DIR),
    )

In [14]:
# Create and populate the Chroma vector database

vectorstore = build_vector_store(chunks)
print(f"Collection: {COLLECTION_NAME}")
print(f"Persist directory: {CHROMA_DIR}")

Collection: logistics_rag
Persist directory: /Users/swateek/Documents/codebase/misc/swateek-github/upgrad-capstone/practice-capstone-project/swateek/chroma/logistics_rag


In [15]:
# Check the number of vectors stored

vector_count = vectorstore._collection.count()
print(f"Vector count: {vector_count}")
print(f"Chunk count: {len(chunks)}")
print(f"Vector count equals chunk count: {vector_count == len(chunks)}")

smoke_retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVAL_K})
smoke_hits = smoke_retriever.invoke("incident management tier definitions")
print(f"Smoke retrieval returned {len(smoke_hits)} documents")

Vector count: 68
Chunk count: 68
Vector count equals chunk count: True
Smoke retrieval returned 4 documents


### **2.2 Create the RAG Chain** <font color=red>[19 marks]</font>

#### **2.2.1 Build the RAG Pipeline** <font color=red>[12 marks]</font>

Form the complete RAG pipeline by connecting the vector database retriever, prompt and language model.

In [16]:
# Additional imports for the RAG chain are already included above
print("RAG imports ready: ChatGoogleGenerativeAI, ChatPromptTemplate, Chroma retriever")

RAG imports ready: ChatGoogleGenerativeAI, ChatPromptTemplate, Chroma retriever


In [17]:
# Format retrieved documents as context


def format_context(retrieved_documents: list[Document]) -> str:
    blocks = []
    for document in retrieved_documents:
        file_name = document.metadata.get("file_name", "unknown")
        blocks.append(f"[Source: {file_name}]\n{document.page_content}")
    return "\n\n".join(blocks)

In [19]:
# Create the RAG chain

SYSTEM_PROMPT = """You are the AeroVelo Logistics Global operations assistant.
Answer using only the CONTEXT below.
If the CONTEXT does not contain enough information, say that the
supplied documents do not contain the answer.
Do not invent document IDs, amounts, roles, or procedures.
If CONTEXT passages disagree, state the disagreement and cite the
file names. Keep answers concise and factual."""

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_PROMPT),
        ("human", "CONTEXT:\n{context}\n\nQUESTION:\n{question}"),
    ]
)

In [21]:
# Initialise the RAG chain


def require_api_key() -> None:
    if not os.environ.get("GOOGLE_API_KEY"):
        raise RuntimeError(
            "GOOGLE_API_KEY is missing. Add GOOGLE_API_KEY or GEMINI_API_KEY to a .env file and re-run the import cell."
        )


require_api_key()
llm = ChatGoogleGenerativeAI(
    model=CHAT_MODEL_NAME,
    temperature=0,
    max_output_tokens=MAX_OUTPUT_TOKENS,
    thinking_level="minimal",
)
retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVAL_K})

print(f"Chat model: {CHAT_MODEL_NAME}")
print("Temperature: 0")
print("thinking_level: minimal")
print(f"max_output_tokens: {MAX_OUTPUT_TOKENS}")
print(f"Retriever k: {RETRIEVAL_K}")
print("Assumptions:")
print(
    "- Packages are installed in the project .venv and this kernel uses that environment."
)
print(f"- Dataset path: {DATASET_DIR}")
print(
    f"- Chroma collection {COLLECTION_NAME} is deleted and recreated at {CHROMA_DIR}."
)
print(f"- Embeddings: {EMBEDDING_MODEL_NAME} (local).")
print("- Gemini chat calls need network access and GOOGLE_API_KEY from .env.")

Chat model: gemini-3.5-flash-lite
Temperature: 0
thinking_level: minimal
max_output_tokens: 512
Retriever k: 4
Assumptions:
- Packages are installed in the project .venv and this kernel uses that environment.
- Dataset path: /Users/swateek/Documents/codebase/misc/swateek-github/upgrad-capstone/practice-capstone-project/swateek/dataset
- Chroma collection logistics_rag is deleted and recreated at /Users/swateek/Documents/codebase/misc/swateek-github/upgrad-capstone/practice-capstone-project/swateek/chroma/logistics_rag.
- Embeddings: sentence-transformers/all-MiniLM-L6-v2 (local).
- Gemini chat calls need network access and GOOGLE_API_KEY from .env.


#### **2.2.2 Generate Answers and Display Sources** <font color=red>[7 marks]</font>

Create a function to generate answers for logistics-related questions.

Use the RAG pipeline to generate an answer and display the source documents used for retrieval.

In [22]:
# Create a function for question answering


def message_text(response) -> str:
    content = response.content
    if isinstance(content, str):
        text = content.strip()
    else:
        parts = []
        for block in content:
            if isinstance(block, str):
                parts.append(block)
            elif isinstance(block, dict):
                block_type = block.get("type")
                if block_type in (None, "text") and block.get("text"):
                    parts.append(block["text"])
            elif getattr(block, "type", None) == "text" and getattr(
                block, "text", None
            ):
                parts.append(block.text)
        text = "\n".join(parts).strip()
    if not text:
        raise RuntimeError(
            "Gemini returned an empty answer. Stop generation and record the failure."
        )
    return text


def ask(question: str) -> dict:
    require_api_key()
    try:
        retrieved = retriever.invoke(question)
        prompt_value = rag_prompt.invoke(
            {"context": format_context(retrieved), "question": question}
        )
        response = llm.invoke(prompt_value)
        answer = message_text(response)
    except RuntimeError:
        raise
    except Exception as exc:
        status = getattr(exc, "status_code", None) or getattr(exc, "code", None)
        message = str(exc)
        lowered = message.lower()
        auth_or_quota = status in (401, 403) or any(
            token in lowered
            for token in ("401", "403", "quota", "permission", "api key")
        )
        if auth_or_quota:
            raise RuntimeError(
                f"Gemini generation failed ({status or 'auth/quota'}): {message}"
            ) from exc
        raise RuntimeError(f"Gemini generation failed: {message}") from exc

    sources = list(
        dict.fromkeys(
            document.metadata.get("file_name", "unknown") for document in retrieved
        )
    )
    return {"answer": answer, "sources": sources, "documents": retrieved}

In [23]:
# Test the RAG assistant with an example logistics question

smoke_question = (
    "What safety override applies when a cycle-expired Valve-B unit "
    "continues to operate above 2,800 psi?"
)
smoke_result = ask(smoke_question)
print("Question:")
print(smoke_question)
print("\nAnswer:")
print(smoke_result["answer"])
print("\nRetrieved source filenames:")
for source in smoke_result["sources"]:
    print(f"- {source}")

/Users/swateek/Documents/codebase/misc/swateek-github/upgrad-capstone/practice-capstone-project/swateek/results/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Question:
What safety override applies when a cycle-expired Valve-B unit continues to operate above 2,800 psi?

Answer:
Continued operation above 2,800 psi on a cycle-expired Valve-B unit constitutes a Tier-3 safety event under SOP OP-204, regardless of the calculated financial impact. This safety-tier override supersedes the financial-impact tier classification in Section 3.1.

Retrieved source filenames:
- BOM-FLT-H2NEX.md
- OP-204_Incident_Management_v4.md
- AVL-PUR-012_Supplier_Qualification.md


## **3. RAG Evaluation** <font color=red>[20 marks]</font>

### **3.1 Evaluation and Inference** <font color=red>[20 marks]</font>

#### **3.1.1 Prepare Evaluation Questions and Reference Answers** <font color=red>[4 marks]</font>

Create a small evaluation set containing logistics questions and their expected ground-truth answers.

In [24]:
# Create evaluation questions and ground-truth answers

evaluation_pairs = [
    {
        "id": "Q1",
        "theme": "Supplier performance",
        "question": "What are the key performance indicator targets for Strategic and Preferred suppliers?",
        "reference_answer": (
            "On-time delivery rate target >95%; quality acceptance rate target >99%; "
            "lead time adherence within \u00b15 days of quoted lead time; incident response time "
            "target <4 hours for critical supply disruptions."
        ),
    },
    {
        "id": "Q2",
        "theme": "Incidents",
        "question": "When does a Tier-2 logistics incident become a Critical Exception, and who must sign off?",
        "reference_answer": (
            "If compounding costs, fines, or penalties of a single Tier-2 incident exceed $10,000 USD, "
            "the incident escalates to a Critical Exception. Critical Exceptions require formal written "
            "sign-off of the respective Regional Operations Vice President."
        ),
    },
    {
        "id": "Q3",
        "theme": "Regulatory reporting",
        "question": (
            "What are the reporting windows and mandatory notification targets for Critical, Major, "
            "and Minor regulatory breaches?"
        ),
        "reference_answer": (
            "Critical: 2 hours \u2014 Regional VP, General Counsel, relevant regulator. "
            "Major: 24 hours \u2014 Regional Operations Director, General Counsel. "
            "Minor: 72 hours \u2014 Regional Operations Director."
        ),
    },
    {
        "id": "Q4",
        "theme": "Supplier data processing",
        "question": "Does emergency procurement from EuroVolt GmbH require a provisional DPA?",
        "reference_answer": (
            "EuroVolt GmbH has a Master DPA on file. Emergency procurement does not require a provisional DPA "
            "when (1) scope is within existing MDPA service categories, (2) no new personal data categories "
            "are introduced, and (3) the EMEA Regional Compliance Officer is notified within 24 hours. "
            "Otherwise the standard 72-hour provisional DPA rule applies."
        ),
    },
    {
        "id": "Q5",
        "theme": "Financial authorisation",
        "question": (
            "Who must approve emergency procurement between $5,001 and $50,000, and what confirmation is required?"
        ),
        "reference_answer": (
            "The Regional Operations Director must approve prior to commitment. "
            "Verbal approval is acceptable; written confirmation is required within 24 hours."
        ),
    },
]

print(f"Evaluation pairs: {len(evaluation_pairs)}")
for item in evaluation_pairs:
    print(f"{item['id']} ({item['theme']}): {item['question']}")

Evaluation pairs: 5
Q1 (Supplier performance): What are the key performance indicator targets for Strategic and Preferred suppliers?
Q2 (Incidents): When does a Tier-2 logistics incident become a Critical Exception, and who must sign off?
Q3 (Regulatory reporting): What are the reporting windows and mandatory notification targets for Critical, Major, and Minor regulatory breaches?
Q4 (Supplier data processing): Does emergency procurement from EuroVolt GmbH require a provisional DPA?
Q5 (Financial authorisation): Who must approve emergency procurement between $5,001 and $50,000, and what confirmation is required?


#### **3.1.2 Evaluate Answer and Retrieval Similarity** <font color=red>[10 marks]</font>

Create a function to evaluate the generated answers.

Evaluate the RAG responses using semantic similarity between the generated answer and ground truth, along with the relevance of the retrieved context.

In [25]:
# Create a function to evaluate the RAG pipeline


def cosine_pair(left, right) -> float:
    return float(cosine_similarity([left], [right])[0][0])


def evaluate_question(item: dict) -> dict:
    result = ask(item["question"])
    answer_vector = embeddings.embed_query(result["answer"])
    reference_vector = embeddings.embed_query(item["reference_answer"])
    answer_similarity = cosine_pair(answer_vector, reference_vector)
    chunk_scores = [
        cosine_pair(reference_vector, embeddings.embed_query(document.page_content))
        for document in result["documents"]
    ]
    retrieval_similarity = max(chunk_scores) if chunk_scores else 0.0
    preview = result["answer"].replace("\n", " ")
    if len(preview) > 180:
        preview = preview[:177] + "..."
    return {
        "id": item["id"],
        "answer_similarity": answer_similarity,
        "retrieval_similarity": retrieval_similarity,
        "sources": ", ".join(result["sources"]),
        "answer_preview": preview,
    }


def evaluate_rag(pairs: list[dict]) -> pd.DataFrame:
    return pd.DataFrame([evaluate_question(item) for item in pairs])

In [26]:
# Evaluate the RAG pipeline

evaluation_results = evaluate_rag(evaluation_pairs)
mean_answer_similarity = float(evaluation_results["answer_similarity"].mean())
mean_retrieval_similarity = float(evaluation_results["retrieval_similarity"].mean())
print(evaluation_results.to_string(index=False))
print()
print(f"Mean answer similarity: {mean_answer_similarity:.4f}")
print(f"Mean retrieval similarity: {mean_retrieval_similarity:.4f}")

/Users/swateek/Documents/codebase/misc/swateek-github/upgrad-capstone/practice-capstone-project/swateek/results/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/Users/swateek/Documents/codebase/misc/swateek-github/upgrad-capstone/practice-capstone-project/swateek/results/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/Users/swateek/Documents/codebase/misc/swateek-github/upgrad-capstone/practice-capstone-project/swateek/results/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling param

id  answer_similarity  retrieval_similarity                                                                                                                   sources                                                                                                                                                                       answer_preview
Q1              0.791                 0.512                                                                                     AVL-PUR-012_Supplier_Qualification.md Based on the supplied documents, the key performance indicator targets for all Strategic and Preferred suppliers are:  * On-time delivery rate: >95% * Quality acceptance rate: >...
Q2              0.841                 0.756         OP-204_Incident_Management_v4.md, RGM-2026_Regional_Governance_Matrix.md, RGM-2026_Regional_Governance_Matrix.pdf A Tier-2 logistics incident becomes a Critical Exception if its compounding costs, fines, or penalties exceed $10,000 USD.   The incident st

#### **3.1.3 Interpret Results and Test Unsupported Questions** <font color=red>[6 marks]</font>

Draw inferences from the evaluation results and test how the RAG system handles a question that is not supported by the provided documents.

Summarise the average evaluation scores and verify that the assistant does not generate unsupported information when the answer is unavailable in the logistics knowledge base.

In [27]:
# Summarise the evaluation results

best_row = evaluation_results.loc[evaluation_results["answer_similarity"].idxmax()]
worst_row = evaluation_results.loc[evaluation_results["answer_similarity"].idxmin()]
print(f"Mean answer similarity: {mean_answer_similarity:.4f}")
print(f"Mean retrieval similarity: {mean_retrieval_similarity:.4f}")
print(
    f"Highest answer similarity: {best_row['id']} "
    f"({best_row['answer_similarity']:.4f}) sources: {best_row['sources']}"
)
print(
    f"Lowest answer similarity: {worst_row['id']} "
    f"({worst_row['answer_similarity']:.4f}) sources: {worst_row['sources']}"
)
print("Limitations:")
print("- RGM Markdown and PDF copies can occupy two of the four retrieval slots.")
print("- FIN-AUTH-101 and RGM-2026 disagree on some monthly authorisation limits.")
print("- Similarity scores are diagnostic. They are not accuracy percentages.")
print(
    "- Runtime: gemini-3.5-flash-lite at temperature 0, embeddings "
    "sentence-transformers/all-MiniLM-L6-v2."
)

Mean answer similarity: 0.7171
Mean retrieval similarity: 0.6449
Highest answer similarity: Q4 (0.9311) sources: LEG-GDPR-007_Supplier_Data_Processing.md, BOM-FLT-H2NEX.md
Lowest answer similarity: Q3 (0.4904) sources: SOP-COM-015_Regulatory_Breach_Report.md, LEG-GDPR-007_Supplier_Data_Processing.md
Limitations:
- RGM Markdown and PDF copies can occupy two of the four retrieval slots.
- FIN-AUTH-101 and RGM-2026 disagree on some monthly authorisation limits.
- Similarity scores are diagnostic. They are not accuracy percentages.
- Runtime: gemini-3.5-flash-lite at temperature 0, embeddings sentence-transformers/all-MiniLM-L6-v2.


In [28]:
# Test an unsupported question

unsupported_question = "What is the annual operating budget allocation for each regional division in AVL-BUD-2026?"
unsupported_result = ask(unsupported_question)
print("Question:")
print(unsupported_question)
print("\nAnswer:")
print(unsupported_result["answer"])
print("\nRetrieved source filenames:")
for source in unsupported_result["sources"]:
    print(f"- {source}")
print("\nThis question is excluded from the supported-question averages.")

/Users/swateek/Documents/codebase/misc/swateek-github/upgrad-capstone/practice-capstone-project/swateek/results/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Question:
What is the annual operating budget allocation for each regional division in AVL-BUD-2026?

Answer:
The supplied documents do not contain the answer.

Retrieved source filenames:
- FIN-AUTH-101_Authorization_Framework.md
- RGM-2026_Regional_Governance_Matrix.md

This question is excluded from the supported-question averages.


## **4. Conclusion** <font color=red>[5 marks]</font>

### **4.1 Conclusions and Insights** <font color=red>[5 marks]</font>

#### **4.1.1 Summarise Findings and Limitations** <font color=red>[5 marks]</font>

Conclude with the results obtained from the data analysis, document chunking, vector retrieval, RAG pipeline and evaluation.

### Data analysis

The loader found 10 documents: 9 Markdown files and 1 PDF, across compliance, finance, governance, HR, legal, operations, and supply chain. After whitespace normalisation, cleaned character length runs from 1,395 (`RGM-2026_Regional_Governance_Matrix.pdf`) to 5,605 (`HR-POL-030-EMEA_Remote_Work.md`), with an average of 4,077.5 characters. Dollar amounts and technical values such as `$10,000` and `2,800 psi` were preserved.

The most frequent non-stop words are operational terms: `regional` (67), `avl` (55), `remote` (54), and `operations` (40). Pairwise TF-IDF cosine similarity is highest between the two remote-work policies (0.879). The RGM Markdown and PDF copies score 0.674. That PDF extract is much shorter than its Markdown twin (1,395 versus 4,973 characters), so the copies are related and the PDF text is incomplete. Most cross-function pairs sit well below these two scores.

### Document chunking and vector retrieval

`RecursiveCharacterTextSplitter` with `chunk_size=800` and `chunk_overlap=120` split the 10 documents into 68 chunks. Each chunk keeps `file_name`, `category`, `file_type`, and `source_path`. Embeddings use `sentence-transformers/all-MiniLM-L6-v2`. The Chroma collection `logistics_rag` is deleted and recreated on each run, then filled once. The stored vector count is 68, equal to the chunk count. A smoke retrieval for incident-tier definitions returned the configured `k=4` documents.

### RAG pipeline

The chain retrieves four chunks, formats each block with its file name, and asks `gemini-3.5-flash-lite` (temperature set to 0, `max_output_tokens` 512) to answer only from that context. On the Valve-B smoke question, the assistant stated that continued operation above 2,800 psi on a cycle-expired Valve-B unit is a Tier-3 safety event under SOP OP-204, and it cited `BOM-FLT-H2NEX.md` and `OP-204_Incident_Management_v4.md`.

### Evaluation

Five supported questions were scored with MiniLM cosine similarity. Mean answer similarity is 0.7171 and mean retrieval similarity is 0.6449.

| Question | Theme | Answer similarity | Retrieval similarity | Sources |
| --- | --- | ---: | ---: | --- |
| Q1 | Supplier performance | 0.791 | 0.512 | `AVL-PUR-012_Supplier_Qualification.md` |
| Q2 | Incidents | 0.841 | 0.756 | `OP-204`, both RGM copies |
| Q3 | Regulatory reporting | 0.490 | 0.546 | `SOP-COM-015`, `LEG-GDPR-007` |
| Q4 | Supplier data processing | 0.931 | 0.876 | `LEG-GDPR-007`, `BOM-FLT-H2NEX.md` |
| Q5 | Financial authorisation | 0.532 | 0.535 | `FIN-AUTH-101`, `LEG-GDPR-007`, `RGM-2026` |

Q4 is the strongest result: the generated answer stays close to the EuroVolt GmbH Master DPA conditions. Q3 is the weakest, even though `SOP-COM-015_Regulatory_Breach_Report.md` was retrieved. The unsupported question about the AVL-BUD-2026 regional operating budget returned “The supplied documents do not contain the answer.” Retrieved filenames were still shown (`FIN-AUTH-101` and `RGM-2026`), and that question was excluded from the averages.

### Limitations

- The RGM Markdown and PDF copies can occupy two of the four retrieval slots. That happened on Q2.
- `FIN-AUTH-101` and `RGM-2026` disagree on some monthly authorisation limits, so a grounded answer can still surface a conflict.
- Answer and retrieval similarity are diagnostic scores. They are not accuracy percentages.
- Generation depends on network access and `GOOGLE_API_KEY`. The Gemini client also warned that `gemini-3.5-flash-lite` ignores the requested temperature and uses fixed sampling defaults.

### Practical improvement

When the RGM Markdown and PDF both rank in the top four, drop or down-rank the duplicate so one near-copy does not consume a slot that could hold a different source. That change follows directly from the Q2 retrieval, where both RGM files were returned together with `OP-204`.